In [ ]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 3 – Location problems
#  Section: Exercise 5
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP        # Modeling language
using HiGHS       # Solver
using CSV         # For reading CSV files
using DataFrames  # For handling DataFrame operations

include("utils/mclp_utils.jl")

# Main function to solve MCLP
function solve_mclp(file_path; radius = 200, p = 7)
    # Load coordinates
    coordinates = CSV.read(file_path, header=true, DataFrame) |> Matrix{Float64}
    
    # Create distance matrix
    D = create_distance_matrix(coordinates)
    
    # Create coverage matrix considering the defined radius
    C = D .< radius

    # Number of demand/facility points
    n = size(C, 1)

    # Create a vector of equal demand for each point
    d = ones(n)

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Variables
    @variable(model, x[1:n], Bin)
    @variable(model, y[1:n], Bin)

    # Objective: minimize number of facilities opened
    @objective(model, Max, sum(d[i]*y[i] for i in 1:n))

    # Constraint: exactly p facilities are opened
    @constraint(model, sum(x) == p)

    # Constraint: each demand point is covered by at least one facility
    @constraint(model, [i in 1:n], sum(C[i,j] * x[j] for j in 1:n) >= y[i])

    # Solve the model
    JuMP.optimize!(model)

    # Get solution vectors
    x_vals = JuMP.value.(x)
    y_vals = JuMP.value.(y)

    # Get indices of selected facilities
    selected_facilities = findall(xi -> xi > 0.5, x_vals)
    covered_points = findall(yi -> yi > 0.5, y_vals)

    # Display solution
    println("Number of facilities opened: ", length(selected_facilities))
    println("Number of points covered: ", length(covered_points), "/", n, " ($(round(100*length(covered_points)/n, digits=2))%)")
    println("Selected facility indices: ", selected_facilities)

    # Display solution on map
    map = plot_solution(selected_facilities, covered_points, coordinates, radius)
    display(map)
end

# Example usage
solve_mclp("data/sjc_coordinates.csv", radius = 400, p = 3)

Number of facilities opened: 3
Number of points covered: 101/102 (99.02%)
Selected facility indices: [7, 40, 73]


PyObject <folium.folium.Map object at 0x000001DB19FA3440>